# 02 — Approximate Nearest Neighbor (ANN) dengan FAISS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/07-embeddings-vectordb/02_ann_faiss_hnsw.ipynb)

Di notebook 01, pencarian dilakukan dengan menghitung skor query terhadap **semua** dokumen (*brute force*). Untuk 1.000 kalimat itu instan, tetapi sistem nyata bisa berisi **jutaan hingga miliaran** vektor. Biayanya sebanding dengan `N × d` per query, sehingga terlalu lambat.

**ANN** (*Approximate Nearest Neighbor*) mengorbankan sedikit akurasi demi kecepatan puluhan hingga ribuan kali lipat. Semua vector database (notebook 03) memakai algoritma ANN di dalamnya.

Isi notebook:
1. Brute force sebagai *ground truth*.
2. **IVF**: kelompokkan vektor dengan k-means, lalu cari hanya di klaster terdekat.
3. **HNSW**: graf navigasi berlapis (dipakai Chroma, Qdrant, pgvector).
4. **PQ**: kompresi vektor untuk menghemat memori.
5. Kurva *trade-off* **recall vs latensi vs memori**.

Library: [`faiss`](https://github.com/facebookresearch/faiss) dari Meta (MIT). Notebook ini tidak mengunduh data apa pun dan tidak butuh GPU.

## 0. Instalasi dan Data Sintetis

Kita buat vektor sintetis 384 dimensi (sama dengan model embedding di notebook 01) yang meniru sifat embedding asli:
- **Berkelompok**, seperti embedding asli yang membentuk topik-topik.
- **Dimensi intrinsik rendah**: walaupun vektornya 384 dimensi, variasinya hanya berada di sebagian kecil arah (di sini 32 dimensi laten + sedikit noise). Embedding dari model sungguhan juga seperti ini, dan sifat inilah yang membuat ANN bekerja baik.

Vektor dinormalisasi sehingga inner product = cosine.

`QUICK_RUN=1` memakai data yang lebih kecil.

In [ ]:
%pip install -q faiss-cpu

Membuat data: `N` vektor database dan `N_QUERY` vektor query dari distribusi yang sama.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import faiss

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
N = 20_000 if QUICK_RUN else 200_000
N_QUERY = 200 if QUICK_RUN else 1_000
D, K = 384, 10
rng = np.random.default_rng(0)


LATEN, N_TOPIK = 32, 200
proyeksi = rng.normal(size=(LATEN, D)).astype("float32")        # ruang laten 32D -> 384D
pusat_topik = rng.normal(size=(N_TOPIK, LATEN)).astype("float32")


def buat_data(n):
    z = pusat_topik[rng.integers(0, N_TOPIK, n)] + 0.8 * rng.normal(size=(n, LATEN)).astype("float32")
    x = z @ proyeksi + 0.15 * np.sqrt(LATEN) * rng.normal(size=(n, D))
    x = (x / np.linalg.norm(x, axis=1, keepdims=True)).astype("float32")   # FAISS butuh float32
    return x


xb = buat_data(N)
xq = buat_data(N_QUERY)
print(f"database: {xb.shape} ({xb.nbytes / 1e6:.0f} MB) | query: {xq.shape}")

## 1. Brute Force (Exact Search)

`IndexFlatIP` menghitung inner product ke **semua** vektor, sama seperti notebook 01 tetapi dioptimasi. Hasilnya menjadi **ground truth** untuk mengukur *recall* metode ANN:

$$\text{recall@}k = \frac{|\text{hasil ANN} \cap \text{hasil exact}|}{k}$$

In [ ]:
def ukur(index, nama, **params):
    t0 = time.perf_counter()
    _, idx = index.search(xq, K)
    ms = (time.perf_counter() - t0) / N_QUERY * 1000
    recall = np.mean([len(set(idx[i]) & set(gt[i])) / K for i in range(N_QUERY)])
    return {"metode": nama, **params, "recall@10": recall, "ms/query": ms}


flat = faiss.IndexFlatIP(D)
flat.add(xb)
t0 = time.perf_counter()
_, gt = flat.search(xq, K)                  # ground truth
ms_flat = (time.perf_counter() - t0) / N_QUERY * 1000

# Bandingkan dengan NumPy murni
t0 = time.perf_counter()
_ = np.argsort(-(xq[:50] @ xb.T), axis=1)[:, :K]
ms_numpy = (time.perf_counter() - t0) / 50 * 1000
print(f"NumPy brute force : {ms_numpy:.2f} ms/query")
print(f"FAISS IndexFlatIP : {ms_flat:.2f} ms/query (recall 1.0 karena exact)")
hasil = [{"metode": "Flat (exact)", "recall@10": 1.0, "ms/query": ms_flat, "MB": xb.nbytes / 1e6}]

## 2. IVF: Inverted File Index

Idenya seperti **indeks buku**:
1. **Training**: jalankan k-means untuk membagi ruang menjadi `nlist` klaster (sel Voronoi).
2. **Menambah data**: setiap vektor dimasukkan ke daftar klaster terdekatnya.
3. **Mencari**: bandingkan query dengan pusat klaster, lalu periksa hanya isi `nprobe` klaster terdekat.

`nprobe` adalah tombol trade-off: nilai kecil → cepat tetapi bisa melewatkan tetangga yang berada di klaster sebelah; nilai besar → akurat tetapi lambat.

In [ ]:
nlist = int(2 * np.sqrt(N))              # aturan praktis: ~√N klaster
quantizer = faiss.IndexFlatIP(D)
ivf = faiss.IndexIVFFlat(quantizer, D, nlist, faiss.METRIC_INNER_PRODUCT)
t0 = time.perf_counter()
ivf.train(xb)
ivf.add(xb)
print(f"IVF: {nlist} klaster, training + add {time.perf_counter() - t0:.1f} detik")

for nprobe in [1, 4, 16, 64]:
    ivf.nprobe = nprobe
    hasil.append({**ukur(ivf, "IVF", nprobe=nprobe), "MB": xb.nbytes / 1e6})
    print(f"  nprobe={nprobe:<3} recall@10 {hasil[-1]['recall@10']:.3f} | {hasil[-1]['ms/query']:.3f} ms/query")

## 3. HNSW: Hierarchical Navigable Small World

HNSW membangun **graf berlapis**: setiap vektor terhubung ke beberapa tetangga terdekatnya.
- Lapisan atas berisi sedikit simpul dengan "jalan tol" jarak jauh; lapisan bawah berisi semua simpul dengan hubungan lokal.
- Pencarian dimulai dari lapisan atas, bergerak rakus (*greedy*) ke simpul yang makin dekat dengan query, lalu turun lapisan demi lapisan. Mirip mencari alamat: negara → kota → jalan → rumah.

Parameter:
- `M`: jumlah tetangga per simpul. Lebih besar → recall lebih tinggi, memori lebih besar.
- `efSearch`: lebar pencarian saat query (tombol trade-off seperti `nprobe`).

HNSW tidak butuh training, mendukung penambahan data kapan saja, dan menjadi algoritma default di hampir semua vector database.

In [ ]:
hnsw = faiss.IndexHNSWFlat(D, 32, faiss.METRIC_INNER_PRODUCT)   # M = 32
hnsw.hnsw.efConstruction = 80
t0 = time.perf_counter()
hnsw.add(xb)
print(f"HNSW: dibangun dalam {time.perf_counter() - t0:.1f} detik")

mb_hnsw = (xb.nbytes + N * 32 * 2 * 4) / 1e6    # vektor + perkiraan link graf
for ef in [16, 32, 64, 128]:
    hnsw.hnsw.efSearch = ef
    hasil.append({**ukur(hnsw, "HNSW", efSearch=ef), "MB": mb_hnsw})
    print(f"  efSearch={ef:<4} recall@10 {hasil[-1]['recall@10']:.3f} | {hasil[-1]['ms/query']:.3f} ms/query")

## 4. PQ: Product Quantization (Kompresi)

Untuk miliaran vektor, **memori** menjadi masalah: 1 miliar × 384 × 4 byte ≈ 1,5 TB. **Product Quantization** memotong setiap vektor menjadi `m` sub-vektor, lalu mengganti setiap sub-vektor dengan ID pusat k-means terdekat (1 byte). Vektor 1.536 byte menjadi hanya `m` byte.

Harganya: jarak menjadi perkiraan, sehingga recall turun. `IVFPQ` menggabungkan IVF (kecepatan) dan PQ (memori).

Perhatikan bahwa recall IVFPQ **berhenti naik** walaupun `nprobe` diperbesar: batasnya bukan lagi jumlah klaster yang diperiksa, melainkan kesalahan jarak akibat kompresi. Solusi umumnya adalah **re-ranking**: ambil kandidat lebih banyak dengan PQ, lalu hitung ulang jaraknya dengan vektor asli (lihat Latihan).

In [ ]:
m = 48                                          # 48 sub-vektor × 8 bit = 48 byte per vektor
ivfpq = faiss.IndexIVFPQ(faiss.IndexFlatIP(D), D, nlist, m, 8, faiss.METRIC_INNER_PRODUCT)
t0 = time.perf_counter()
ivfpq.train(xb)
ivfpq.add(xb)
print(f"IVFPQ dibangun dalam {time.perf_counter() - t0:.1f} detik | {D * 4} → {m} byte per vektor ({D * 4 // m}× lebih kecil)")

for nprobe in [4, 16, 64]:
    ivfpq.nprobe = nprobe
    hasil.append({**ukur(ivfpq, "IVFPQ", nprobe=nprobe), "MB": N * m / 1e6})
    print(f"  nprobe={nprobe:<3} recall@10 {hasil[-1]['recall@10']:.3f} | {hasil[-1]['ms/query']:.3f} ms/query")

## 5. Trade-off: Recall vs Latensi vs Memori

Tidak ada metode yang terbaik di semua aspek. Pilih berdasarkan kebutuhan:
- **< 100 ribu vektor**: Flat (exact) sering sudah cukup cepat. Jangan menambah kompleksitas.
- **Jutaan vektor, memori cukup**: HNSW (recall tinggi, cepat).
- **Ratusan juta+ vektor**: IVFPQ atau varian lain dengan kompresi.

In [ ]:
import pandas as pd

tabel = pd.DataFrame(hasil)
print(tabel.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
for metode, g in tabel.groupby("metode", sort=False):
    ax.plot(g["ms/query"], g["recall@10"], "o-", label=metode)
ax.set_xscale("log"); ax.set_xlabel("latensi (ms/query, skala log)"); ax.set_ylabel("recall@10")
ax.set_title(f"Trade-off ANN ({N:,} vektor, {D} dimensi)"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 6. Menyimpan dan Memuat Index

Membangun index besar bisa memakan waktu lama, jadi simpan hasilnya ke disk. (Di Colab, simpan ke Google Drive agar tidak hilang.)

In [ ]:
faiss.write_index(hnsw, "hnsw.index")
dimuat = faiss.read_index("hnsw.index")
hnsw.hnsw.efSearch = dimuat.hnsw.efSearch = 64          # parameter pencarian harus sama agar adil
_, idx_a = hnsw.search(xq[:5], K)
_, idx_b = dimuat.search(xq[:5], K)
print(f"ukuran file: {os.path.getsize('hnsw.index') / 1e6:.0f} MB | hasil sama setelah dimuat ulang? {np.array_equal(idx_a, idx_b)}")

## Ringkasan

- Brute force akurat tetapi biayanya `O(N·d)` per query; ANN menukar sedikit recall dengan kecepatan jauh lebih tinggi.
- **IVF**: k-means + cari di `nprobe` klaster terdekat.
- **HNSW**: graf berlapis, pencarian greedy; `efSearch` mengatur trade-off. Ini default di vector DB.
- **PQ**: kompresi vektor puluhan kali lipat dengan harga recall.
- Selalu ukur **recall@k terhadap exact search** saat memilih parameter.

## Latihan

1. **Parameter M.** Bangun HNSW dengan `M` = 8, 16, 32, dan 64. Untuk setiap `M`, cari `efSearch` terkecil yang mencapai recall@10 ≥ 0.95, lalu catat latensi, waktu build, dan ukuran file index. Buat tabel dan simpulkan nilai `M` yang paling seimbang.
2. **Data asli.** Ganti data sintetis dengan embedding asli: encode 10.000+ kalimat (misalnya gabungan NusaX semua bahasa, atau dataset teks lain) dengan model dari notebook 01, lalu ulangi perbandingan Flat vs IVF vs HNSW. Apakah kurvanya berbeda dari data sintetis? Mengapa sebaran data berpengaruh?
3. **Re-ranking & skala.** (a) Bungkus IVFPQ dengan `faiss.IndexRefineFlat` (ambil `k_factor` × k kandidat dengan PQ, lalu hitung ulang dengan vektor asli) dan ukur kenaikan recall serta latensinya. (b) Perkirakan memori untuk 10 juta dan 100 juta vektor 384 dimensi dengan Flat, HNSW (M = 32), dan IVFPQ (m = 48). Mana yang muat di RAM Colab (~12 GB)?